<h1 align="center"><strong><font size="6"> SFM II — Expanding-Window Harness </font></strong></h1>

<br>

**Purpose (referee report M4):** re-run model selection on a repaired, deployment-shaped harness —
train through season *t*, score season *t+1* **frozen**, compare variants with **paired tests on
identical player-appearances**. Direct descendant of `SFM_II__dev.ipynb` (same data prep, same
model cells, revised 2026-08-04) with the fold loop, ledger, and verdict machinery of
`SFMMO__dev_EW.ipynb`.

**Workflow (as in the SFMMO harness):** pick ONE `devVersion` in USER INTERACTION → run top to
bottom → the Export cell writes `Evaluation__SFM_II_Dev<V>__scaleCS__EW.pkl`. Repeat per variant.
The **Cross-Variant Verdict** cell reads every exported ledger and applies the pre-registered rule:
*a challenger replaces the incumbent (`C`) only if it wins **both** log-loss and RPS with paired
|t| ≥ 2.5 pooled across folds; ties resolve to parsimony.* (Rule fixed 2026-08-04, before any run.)

**Holdout:** `2024/25` — sealed by asserts, scored ONCE via `RUN_HOLDOUT` for the committed spec.
*Disclosure:* 2024/25 was consulted by the legacy dev runs before this harness existed; sealed
forward by decision of 2026-08-04. Append 2025/26 as a pristine second holdout when it lands.

⚠️ **First run per variant: leave `SMOKE = True`** (last fold only, 200/200 draws — the
numpyro/JAX backend is what must pass; SFMMO lesson). Then `SMOKE = False`: ~3 fits/variant
≈ 2.5–3 GPU-h; full 6-variant grid ≈ 15–18 GPU-h.

In [ ]:
# --- Connect to Google-Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -qq pymc numpyro
!pip install -qq arviz
!pip install -qq jax[cuda] -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html

In [ ]:
# --- Usual Libraries:
import copy
import gc
import time
import glob
import os

import numpy as np
import pandas as pd

# --- PyMC & Affiliates:
import arviz as az
import pymc as pm
import pytensor.tensor as pt
import xarray as xr

# --- Stats:
from scipy.stats import norm
from types import SimpleNamespace

seed = sum(map(ord, "sfm-ew"))
rng = np.random.default_rng(seed)

In [ ]:
pm.__version__

In [ ]:
# -------------------------------------- USER INTERACTION -------------------------------------- #

# --- Which GPU did Colab give us? Sampling speed varies ~2.5x between T4 and A100, which is
# --- the difference between a ~18h and a ~45h grid. Recorded so timings are interpretable.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'no GPU'

# --- Set the directory to the datafile ('data_byPlayer__SFM_II.csv'):
directory = '/content/drive/MyDrive/Colab Notebooks/51_SoccerAnalytics/'

# --- Which Dev-Version? (registry in '00 Auxiliaries' -- referee M8.2). ONE variant per run:
# ---   OG        : the PUBLISHED model (Andorra & Goebel 2024) -- points_diff + home_pitch
# ---               only, inter+intra HSGPs. The incumbent: every richer spec must beat it.
# ---   C, F, K   : the evaluation report's picks, re-tried on the repaired harness
# ---   F_MOMs    : F + momentum cross-sectionally standardized     (referee M1.1)
# ---   F_ERA     : F + shared calendar-era HSGP                    (referee M6)
# ---   F_LEAGUE  : F + zero-sum league intercepts (Ligue-1 is now the thinnest league)
# ---   C_MOMs    : C + standardized momentum, SAME HSGPs -- the clean +/-MOM cell
# ---               (reviewer variant 2026-08-11; the original grid confounded MOM with HSGP config)
# ---   A, A_MOMs : momentum (raw / standardized) with NO GPs at all -- the old DevA cell
# ---   C_ELO     : C + ELO-difference (SFMMO-family ratings; cold-start hypothesis on record)
# ---   A_ELO     : A + ELO-difference -- ELO x momentum with no GPs at all
# ---   C_ELO_noGP: ELO + factors, NO momentum, NO GPs -- the momentum-passenger test
# ---               (vs A_ELO) and the pure GP ablation (vs C_ELO)
# --- (F_GKu retired 2026-08-10: positions fully populated -> no Unknown class to flag.)
devVersion = 'C'

# --- Incumbent for the pre-registered Cross-Variant Verdict.
# --- OG (the published SFM I) carries the incumbency: it is the model in the paper and the
# --- most parsimonious spec here, so the burden of proof sits with the richer challengers --
# --- the same logic the SFMMO re-trial used. (C's apparent incumbency came from the old
# --- evaluation report, whose selection ran while defects M2/M3 were active.)
# --- Set BEFORE any real fit and do not change after results exist.
INCUMBENT = 'OG'

# --- SMOKE mode: LAST fold only, 200/200 draws -- a pipeline test on the numpyro backend
# --- (the backend is what must pass; SFMMO lesson). FIRST RUN MUST BE SMOKE. Export refuses.
SMOKE = True

# --- Chains: 4 vectorized chains (decided 2026-08-10 after the replicate experiment --
# --- one of two identical-config runs had min_ess 146 < the 200 gate; 4 chains doubles
# --- total draws and the ESS floor, and makes rhat estimates trustworthy). GPU-vectorized,
# --- so wall-time cost is well under 2x on the A100.
N_chains = 4

# --- Sampler acceptance target. Contingency ladder: 0.9 -> 0.95 -> 0.99 / non-center the
# --- player ZSN. RUNG 2 ENGAGED 2026-08-10: OG (the SMALLEST model) showed 20 divergences
# --- at full draws under 0.9, so 0.9 is retired for this re-trial. Must be IDENTICAL across
# --- all variants (geometry, not spec). If OG is still dirty at 0.95 -> non-center the ZSN.
TARGET_ACCEPT = 0.95

# --- RUNG 3: NEVER TESTED -- and retired anyway. CORRECTED RECORD (2026-08-10): the
# --- intended non-centered experiment accidentally ran with this flag False, producing a
# --- second CENTERED fit of OG fold 1 @ 0.95. The two identical-config replicates gave
# ---   replicate A: 27 divergences, max_rhat 1.011, min_ess 461
# ---   replicate B:  1 divergence,  max_rhat 1.071, min_ess 146
# --- i.e. RUN-TO-RUN VARIANCE (runtime/XLA/GPU) dominates these diagnostics, so a
# --- single-run A/B test of parameterizations was never going to be interpretable.
# --- LADDER CLOSED: centered @ 0.95, N_chains = 4 (doubles the ESS floor so a
# --- replicate-B-style draw still passes the gate, and stabilizes rhat), divergence
# --- rate 0-1.4%/fold documented-accepted for selection. LEAVE False.
NONCENTER_ZSN = False

# --- Expanding-window folds: train through trainT[i], score valT[i] FROZEN:
dict_EW = {'trainT': ['2020/21', '2021/22', '2022/23'],
           'valT':   ['2021/22', '2022/23', '2023/24']}

# --- HOLDOUT (referee M4): season(s) NO experiment may ever tune against. Scored EXACTLY
# ---   ONCE, for the committed spec. Disclosure: 2024/25 was consulted by the legacy dev
# ---   runs (~24 variants) BEFORE this harness existed; sealed forward by decision of
# ---   2026-08-04. When 2025/26 data lands, append it here as a second, pristine holdout.
HOLDOUT_SEASONS = ['2024/25']

# --- THE ONE HOLDOUT RUN: after the Verdict cell + collaborator review, set COMMITTED to the
# ---   selected spec; RUN_HOLDOUT = True re-points the window at 2024/25 and runs it ONCE.
COMMITTED   = 'C_ELO'          # --- committed 2026-08-13; holdout burned 2026-08-14
RUN_HOLDOUT = False

# --- PRODUCTION RUN (mirrors the SFMMO analyst's SFMMOwm__final_EW pattern): fit the
# --- COMMITTED spec on the FULL window -- no validation season, nothing scored -- and write
# --- the model bundle consumed by 006_040__Predictions_ScoringProb. Selection and holdout
# --- are already closed; this run produces the artefact, not evidence.
RUN_PRODUCTION = False
PRODUCTION_TRAIN_END = '2025/26'      # --- last COMPLETE season to train on

if RUN_PRODUCTION:
    # --- SMOKE is deliberately ALLOWED here. The production branch (valT = [None]) skips
    # --- the entire scoring section, so it needs its own pipeline test -- and the hard
    # --- `assert not SMOKE` lives in the Export cell, where it actually matters. A smoke
    # --- run therefore executes this branch end-to-end and then REFUSES to write a
    # --- 200/200-draw bundle: that AssertionError at Export is the smoke test PASSING.
    assert not RUN_HOLDOUT,  'production and holdout are separate runs -- do one at a time'
    assert COMMITTED is not None and COMMITTED == devVersion, 'production fits ONLY the committed spec'
    # --- one "fold": train through PRODUCTION_TRAIN_END, score nothing.
    dict_EW = {'trainT': [PRODUCTION_TRAIN_END], 'valT': [None]}
    print('=' * 95)
    print(f'==  PRODUCTION RUN -- fitting Dev{COMMITTED} on ALL seasons <= {PRODUCTION_TRAIN_END}.')
    if SMOKE:
        print('==  SMOKE: 200/200 draws, branch test only -- the Export cell WILL refuse (that is the pass).')
    print('==  Nothing is scored. Run the "Export -- Production Model" cell afterwards.')
    print('=' * 95)

elif RUN_HOLDOUT:
    assert COMMITTED is not None and COMMITTED == devVersion, 'the holdout runs ONLY the committed spec'
    assert not SMOKE, 'the holdout must run with full draws'
    print('!' * 95)
    print(f'!!  HOLDOUT RUN -- burning {HOLDOUT_SEASONS} ONCE for the committed spec Dev{COMMITTED}.')
    print('!' * 95)
    dict_EW = {'trainT': [max(dict_EW['valT'])], 'valT': list(HOLDOUT_SEASONS)}
else:
    assert not (set(HOLDOUT_SEASONS) & set(dict_EW['valT'])),   'HOLDOUT leaked into the validation folds!'
    assert not (set(HOLDOUT_SEASONS) & set(dict_EW['trainT'])), 'HOLDOUT leaked into the training-end list!'
    assert max(dict_EW['valT']) < min(HOLDOUT_SEASONS),         'a validation season sits at/after the HOLDOUT!'

# -------------------------------------- USER INTERACTION -------------------------------------- #

<br>

## 00 &emsp; Auxiliaries

In [ ]:
# ======================================== Evaluation Auxiliaries ======================================== #
# --- Repaired metrics layer (referee M3): per-row proper scores, ECE with a CLOSED last bin,
# --- RPS normalized by (K-1) so it lives in [0, 1] (convention shared with the SFMMO ledger).


def perrow_logloss(probs, y, eps=1e-12):
    p = np.clip(np.asarray(probs)[np.arange(len(y)), np.asarray(y)], eps, 1.0)
    return -np.log(p)


def perrow_rps(probs, y, n_classes=4):
    cum = np.cumsum(np.asarray(probs), axis=1)
    ycum = (np.arange(n_classes)[None, :] >= np.asarray(y)[:, None]).astype(float)
    return ((cum - ycum) ** 2).sum(axis=1) / (n_classes - 1)


def multi_class_brier_score(probs, y, n_classes=4):
    onehot = np.zeros((len(y), n_classes))
    onehot[np.arange(len(y)), np.asarray(y)] = 1
    return float(np.mean(np.sum((np.asarray(probs) - onehot) ** 2, axis=1)))


def expected_calibration_error(probs, y, n_bins=10):
    # --- M3.2 FIX: the last bin is CLOSED at 1.0 (the old half-open binning dropped
    # --- perfect-confidence predictions from EVERY bin).
    probs = np.asarray(probs)
    confidence = np.max(probs, axis=1)
    pred_classes = np.argmax(probs, axis=1)
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        if i < n_bins - 1:
            in_bin = (confidence >= bin_edges[i]) & (confidence < bin_edges[i + 1])
        else:
            in_bin = (confidence >= bin_edges[i]) & (confidence <= bin_edges[i + 1])
        if np.sum(in_bin) > 0:
            ece += np.abs(confidence[in_bin].mean()
                          - (pred_classes[in_bin] == np.asarray(y)[in_bin]).mean()) * np.sum(in_bin)
    return float(ece / len(y))


def summarize_probs(probs, y):
    return dict(n=int(len(y)),
                logloss=float(perrow_logloss(probs, y).mean()),
                rps=float(perrow_rps(probs, y).mean()),
                brier=multi_class_brier_score(probs, y),
                ece=expected_calibration_error(probs, y),
                acc=float((np.argmax(np.asarray(probs), axis=1) == np.asarray(y)).mean()))


def update_elo(r_home, r_away, result, K=20, home_adv=50):
    # --- SFMMO family ELO (K=20, home_adv=50). NOTE: intent-faithful port -- the SFMMO
    # --- in-notebook version processes BOTH perspective rows per match (double updates,
    # --- home_adv on both sides); here: ONE update per match, home side correctly attributed.
    exp_home = 1 / (1 + 10 ** ((r_away - r_home - home_adv) / 400))
    s_home = {2: 1.0, 1: 0.5, 0: 0.0}[result]
    return r_home + K * (s_home - exp_home), r_away + K * ((1 - s_home) - (1 - exp_home))


def paired_t(diff):
    # --- t of mean(diff) vs 0; diff = challenger - incumbent per row (negative favours challenger)
    d = np.asarray(diff, dtype=float)
    se = d.std(ddof=1) / np.sqrt(len(d))
    return float(d.mean() / se) if se > 0 else float('nan')


# ------------------------ Lengthscale Priors (preliz-free; SFMMO lesson) ------------------------ #
# --- InverseGamma(alpha, beta) constants precomputed OFFLINE (2026-08-04) with the sfmII
# --- venv's preliz 0.23.0: pz.maxent(pz.InverseGamma(), lower, upper) -- bit-identical to
# --- what the notebooks always used. preliz is NOT installed or imported on Colab: it
# --- crashes the pymc/numpyro stack there (known problem from the SFMMO).
LS_PRIORS = {
    'short':  SimpleNamespace(lower=2,  upper=5,  alpha=13.29478010, beta=43.62832634),   # days
    'medium': SimpleNamespace(lower=15, upper=30, alpha=22.45420627, beta=487.52137343),  # days
    'long':   SimpleNamespace(lower=2,  upper=6,  alpha=9.54036477,  beta=34.71244771),   # seasons
    'era':    SimpleNamespace(lower=5,  upper=15, alpha=9.54046944,  beta=86.78264042),   # seasons (F_ERA)
}

In [ ]:
# ======================================== The Variant Registry ======================================== #
# --- Referee M8.2: ONE source of truth. factors_CS / other_factors exactly as SFM_II__dev cell
# --- 'Define the Factors'; mom = pooled-beta momentum; mom_sdz = standardized-MOM challenger;
# --- gk_dummy = GK/Unknown class (RETIRED -- see the guard in the data cell); hsgp in
# --- {'long','both'}; era = calendar-period HSGP; league = zero-sum league intercepts.

# --- POSITION DUMMIES (2026-08-10 vintage): ONE dummy only. Positions are now fully
# --- populated and the unlabelled goalkeeper-like class (the de facto reference group,
# --- ~10% of rows in earlier vintages) is gone, so MID and FOR partition 99.88% of the
# --- data: corr = -0.9975, collinear with the intercept, only their DIFFERENCE identified.
# --- position_FOR alone spans the same space; reference = midfield (+ ~0.1% defenders).

ALL_CS = ['points_diff', 'goalsscored_cum_player', 'goalsconceded_rank_opp',
          'goalsscored_share_player_team', 'goal_appeal']

VARIANTS = {
    # --- the published SFM I (Andorra & Goebel 2024): points_diff + home_pitch, inter+intra
    # --- HSGPs, no momentum, NO position dummies. Kept faithful to the paper -- this is the
    # --- benchmark that says whether the post-paper factor engineering earned its keep.
    'OG':     dict(factors_CS=['points_diff'],
                   other_factors=['home_pitch'],
                   mom=False, mom_sdz=False, gk_dummy=False, hsgp='both', era=False, league=False),
    'C':      dict(factors_CS=ALL_CS,
                   other_factors=['home_pitch', 'position_FOR'],
                   mom=False, mom_sdz=False, gk_dummy=False, hsgp='both', era=False, league=False),
    'F':      dict(factors_CS=ALL_CS,
                   other_factors=['home_pitch', 'position_FOR'],
                   mom=True, mom_sdz=False, gk_dummy=False, hsgp='long', era=False, league=False),
    'K':      dict(factors_CS=['points_diff', 'goalsconceded_rank_opp'],
                   other_factors=['home_pitch', 'position_FOR'],
                   mom=True, mom_sdz=False, gk_dummy=False, hsgp='both', era=False, league=False),
    'F_MOMs': dict(factors_CS=ALL_CS,
                   other_factors=['home_pitch', 'position_FOR'],
                   mom=True, mom_sdz=True, gk_dummy=False, hsgp='long', era=False, league=False),
    'F_LEAGUE': dict(factors_CS=ALL_CS,
                   other_factors=['home_pitch', 'position_FOR'],
                   mom=True, mom_sdz=False, gk_dummy=False, hsgp='long', era=False, league=True),
    'F_ERA':  dict(factors_CS=ALL_CS,
                   other_factors=['home_pitch', 'position_FOR'],
                   mom=True, mom_sdz=False, gk_dummy=False, hsgp='long', era=True, league=False),
    # --- reviewer variant (2026-08-11): the CLEAN +/-momentum contrast. The original grid
    # --- confounded MOM with the HSGP config (C = no-MOM/both, F = MOM/long-only). C_MOMs
    # --- = C + standardized momentum at IDENTICAL HSGP config; standardized, because raw
    # --- MOM's early-season seed ramp is a gameday profile that f_within partially absorbs.
    # --- Pre-registered: replaces C only if it beats C on BOTH metrics at |t| >= 2.5
    # --- (paired, pooled folds); a tie keeps C (parsimony).
    'C_MOMs': dict(factors_CS=ALL_CS,
                   other_factors=['home_pitch', 'position_FOR'],
                   mom=True, mom_sdz=True, gk_dummy=False, hsgp='both', era=False, league=False),
    # --- reviewer variants (2026-08-11, momentum design completion): momentum with NO GPs
    # --- at all (the old DevA cell, never re-run on the repaired harness). If the GPs were
    # --- absorbing momentum's signal, these are the cells where it shows. hsgp='none' is a
    # --- dedicated code path: alpha = player effect only.
    'A':      dict(factors_CS=ALL_CS,
                   other_factors=['home_pitch', 'position_FOR'],
                   mom=True, mom_sdz=False, gk_dummy=False, hsgp='none', era=False, league=False),
    'A_MOMs': dict(factors_CS=ALL_CS,
                   other_factors=['home_pitch', 'position_FOR'],
                   mom=True, mom_sdz=True, gk_dummy=False, hsgp='none', era=False, league=False),
    # --- reviewer variant (2026-08-11): ELO -- the SFMMO's core 'class' covariate, never in
    # --- the SFM. elo_diff is CS-standardized like the other factors. Cold-start hypothesis
    # --- on the record BEFORE the run: points_diff is all-zero at gameday 1 while
    # --- sd(elo_diff)=130 there, so ELO's edge (if any) should concentrate in gd 1-2 buckets.
    'C_ELO':  dict(factors_CS=ALL_CS + ['elo_diff'],
                   other_factors=['home_pitch', 'position_FOR'],
                   mom=False, mom_sdz=False, gk_dummy=False, hsgp='both', era=False, league=False),
    # --- reviewer variant (2026-08-11): ELO x momentum, no GPs. A_ELO vs A isolates ELO's
    # --- increment in momentum's presence; A_ELO vs C_ELO asks whether the GPs still matter
    # --- once ELO carries the strength signal. Raw MOM (one-knob contrast vs A; the raw-MOM
    # --- seed-ramp concern is f_within-specific and there are no GPs here).
    'A_ELO':  dict(factors_CS=ALL_CS + ['elo_diff'],
                   other_factors=['home_pitch', 'position_FOR'],
                   mom=True, mom_sdz=False, gk_dummy=False, hsgp='none', era=False, league=False),
    # --- reviewer variant (2026-08-13): the MOMENTUM-PASSENGER cell. Identical to A_ELO
    # --- minus momentum (equivalently: the champion C_ELO minus its GPs). Two clean
    # --- contrasts, pre-registered BEFORE the run:
    # ---   A_ELO vs C_ELO_noGP : momentum's contribution inside the lean architecture --
    # ---                         if |t| < 2.5, "strength + form" has no measured form part;
    # ---   C_ELO vs C_ELO_noGP : the PURE GP ablation at fixed factors (the A_ELO version
    # ---                         of this contrast had momentum confounded on one side).
    'C_ELO_noGP': dict(factors_CS=ALL_CS + ['elo_diff'],
                   other_factors=['home_pitch', 'position_FOR'],
                   mom=False, mom_sdz=False, gk_dummy=False, hsgp='none', era=False, league=False),
}

cfg = VARIANTS[devVersion]

# --- Resolve the factor lists for this run:
other_factors = list(cfg['other_factors']) + (['position_GKu'] if cfg['gk_dummy'] else [])
factors_CS = list(cfg['factors_CS'])
factors_team = other_factors + factors_CS
if cfg['mom']:
    factors_player = ['goalsscored_MOM_player_sdz' if cfg['mom_sdz'] else 'goalsscored_MOM_player']
else:
    factors_player = []

print(f'Dev{devVersion}:  factors_team = {factors_team}')
print(f'          factors_player = {factors_player}   hsgp = {cfg["hsgp"]}   era = {cfg["era"]}')

<br>

## 0 &emsp; Data Preparation

Identical to `SFM_II__dev.ipynb` (post-revision): verified-clean momentum, within-group position
fill (M5-addendum fix) + GK/Unknown dummy, `season_nbr` on the full window, and — new here (M8.1)
— cross-sectional standardization in **season × gameday** buckets, factor-wise, so train and
frozen-validation rows are scaled by identical machinery.

In [ ]:
# ======================================== Load the Data ======================================== #

data_all = pd.read_csv(f"{directory}/10_data/106_Website/data_byPlayer__SFM_II.csv")

# --- Pre-Processing, Part I:
data_all['kick_off'] = pd.to_datetime(data_all['kick_off'])
data_all = data_all.sort_values(['name_player', 'season', 'kick_off'])

# --- gameday: use the column; ASSERT it agrees with the id_match parse (referee minor)
_parsed = data_all['id_match'].apply(lambda i: int(float(i.split('_')[1][2:])))
data_all['gameday'] = data_all['gameday'].astype(int)
assert (data_all['gameday'] == _parsed).all(), 'gameday column disagrees with id_match parse'


# ======================================== Feature Engineering ======================================== #

# --- Goal Appeal (of the match):
data_all['goal_appeal'] = data_all['goalsconceded_rank_opp'] - data_all['goalsscored_rank_team']


# ------------------------ Player Momentum: EWMA of previous goals ------------------------ #
# --- Timing VERIFIED pre-match (referee M1) -- construction ported unchanged from SFM_II__dev.

data_all['FD__goalsscored_cum_player'] = data_all.groupby(['season', 'name_player'])['goalsscored_cum_player'].diff().fillna(data_all['goalsscored_cum_player'])
playerMomentum = pd.DataFrame(data_all.groupby(['season', 'name_player'])['FD__goalsscored_cum_player'].ewm(halflife=1).mean().reset_index()).set_index('level_2')
data_all['goalsscored_MOM_player'] = playerMomentum['FD__goalsscored_cum_player']


# ------------------------ Positions: WITHIN-group fill (M5-addendum fix) ------------------------ #
# --- The old chained groupby-bfill().ffill() leaked positions ACROSS players (10% of rows).

data_all['position_player'] = data_all.groupby(['season', 'name_player'])['position_player'].transform(lambda s: s.bfill().ffill())
# --- ONE dummy, reference = midfield (+ the ~0.1% defenders) -- see the registry note on
# --- the MID/FOR collinearity introduced by the fully-populated 2026-08-10 positions.
data_all['position_FOR'] = np.where(data_all['position_player'] == 'Sturm', 1, 0)
data_all['position_GKu'] = np.where(data_all['position_player'].isna(), 1, 0)   # --- GK/Unknown (F_GKu)

# --- F_GKu is only a real experiment while an unlabelled position class EXISTS. On a
# --- fully-populated vintage position_GKu is all-zero and F_GKu silently duplicates F.
if cfg['gk_dummy'] and data_all['position_GKu'].sum() == 0:
    raise AssertionError(
        'F_GKu requested but positions are 100% populated -- position_GKu is all-zero, so '
        'this variant is a duplicate of F. Pick a different challenger for the slot.')


# ======================================== Pre-Processing: Part II ======================================== #

print(f"Number of Observations 'goalsscored_share_player_team' > 1: {sum(data_all['goalsscored_share_player_team'] > 1)}")
data_all = data_all.loc[data_all['goalsscored_share_player_team'] <= 1, :]

# --- Some Type-Setting:
data_all['goal'] = data_all['goal'].astype(int)
data_all['goals_in_match'] = data_all['goals_in_match'].astype(int)
data_all['kick_off'] = pd.to_datetime(data_all['kick_off'], yearfirst=True).dt.normalize()

# --- Player-Maturity: season number for each player (computed on the FULL window, pre-split,
# --- so frozen-validation players keep their career index):
data_all['season_nbr'] = data_all.groupby(['name_player'])['season'].transform(lambda x: x.factorize(sort=True)[0])

# --- Calendar-season index (F_ERA): fixed 0..24 mapping over the full window:
data_all['cal_season_nbr'] = data_all['season'].factorize(sort=True)[0]

# --- League index (F_LEAGUE): FIXED ordering from the full window so fold indices are stable:
leagues_ordered = np.sort(data_all['name_league'].unique())
data_all['league_nbr'] = pd.Categorical(data_all['name_league'], categories=leagues_ordered).codes
assert (data_all['league_nbr'] >= 0).all()


# -------------------------------------------- ELO Ratings (C_ELO) -------------------------------------------- #
# --- SFMMO-family conventions: init 1500; season-start regress 0.75r+0.25*1500 for returning
# --- teams, 1300 for promoted; K=20, home_adv=50. Fold-safe by construction: the rating
# --- attached to a match uses only PRIOR results (assigned pre-update). Validated locally
# --- 2026-08-11 on the frozen vintage: 39,931 matches; elo_diff(home) == -elo_diff(away)
# --- exactly; sane top-8; sd(elo_diff)=130 at gameday 1 where points_diff is all-zero.

_mt = (data_all[data_all.home_pitch == 1]
       .drop_duplicates('id_match')[['id_match','name_league','season','kick_off','name_team',
                                     'name_opp','goalsscored_inGame_team','goalsscored_inGame_opp']]
       .rename(columns={'name_team':'home','name_opp':'away',
                        'goalsscored_inGame_team':'g_home','goalsscored_inGame_opp':'g_away'}))
_missing = set(data_all.id_match.unique()) - set(_mt.id_match)
if _missing:  # matches observed only from the away perspective
    _aw = (data_all[data_all.id_match.isin(_missing) & (data_all.home_pitch == 0)]
           .drop_duplicates('id_match')[['id_match','name_league','season','kick_off','name_opp',
                                         'name_team','goalsscored_inGame_opp','goalsscored_inGame_team']])
    _aw.columns = _mt.columns
    _mt = pd.concat([_mt, _aw], ignore_index=True)

_mt['res_home'] = np.select([_mt.g_home > _mt.g_away, _mt.g_home < _mt.g_away], [2, 0], default=1)
_mt['elo_home'] = np.nan
_mt['elo_away'] = np.nan

elo_state = {}   # --- final per-league ratings AFTER the last played match (serving needs these)
for _ll in _mt.name_league.unique():
    _teams = set(_mt.loc[_mt.name_league == _ll, 'home']) | set(_mt.loc[_mt.name_league == _ll, 'away'])
    _ELO = {t: 1500.0 for t in _teams}
    _seasons = sorted(_mt.loc[_mt.name_league == _ll, 'season'].unique())
    for _si, _ss in enumerate(_seasons):
        _sub = _mt[(_mt.name_league == _ll) & (_mt.season == _ss)].sort_values(['kick_off', 'id_match'])
        if _si > 0:
            _prev = set(_mt.loc[(_mt.name_league == _ll) & (_mt.season == _seasons[_si-1]), 'home']) | \
                    set(_mt.loc[(_mt.name_league == _ll) & (_mt.season == _seasons[_si-1]), 'away'])
            for _t in set(_sub.home) | set(_sub.away):
                _ELO[_t] = _ELO[_t] * 0.75 + 1500 * 0.25 if _t in _prev else 1300.0
        for _idx in _sub.index:
            _h, _a = _mt.at[_idx, 'home'], _mt.at[_idx, 'away']
            _mt.at[_idx, 'elo_home'], _mt.at[_idx, 'elo_away'] = _ELO[_h], _ELO[_a]   # PRE-match
            _ELO[_h], _ELO[_a] = update_elo(_ELO[_h], _ELO[_a], _mt.at[_idx, 'res_home'])

    elo_state[_ll] = dict(_ELO)   # --- snapshot after this league's last match

data_all = data_all.merge(_mt[['id_match', 'elo_home', 'elo_away']], on='id_match', how='left')
data_all['elo_team'] = np.where(data_all.home_pitch == 1, data_all.elo_home, data_all.elo_away)
data_all['elo_opp'] = np.where(data_all.home_pitch == 1, data_all.elo_away, data_all.elo_home)
data_all['elo_diff'] = data_all['elo_team'] - data_all['elo_opp']
assert data_all['elo_diff'].notna().all()
elo_match_table = _mt[['id_match', 'name_league', 'season', 'home', 'away', 'elo_home', 'elo_away']].copy()

# --- Winsorization of Goals >= 3 (category-safe under the ordered likelihood):
data_all['goals_cats'] = np.where(data_all['goals_in_match'] >= 3, 3, data_all['goals_in_match'])

# --- Final Sorting for Convenience:
data_all = data_all.sort_values(['name_player', 'kick_off']).reset_index(drop=True)
data_all.shape

In [ ]:
# ==================== Cross-Sectional Standardization (by Season x Gameday) ==================== #
#
# Referee M8.1: ONE definition, symmetric between train and score time. Buckets are
# season x gameday, so they NEVER cross seasons -- training rows are scaled from their own
# season's cross-section and validation rows from theirs, with identical machinery.
# Factor-wise (the old rule zeroed ALL factors in a bucket if ANY single one had zero std).

def _cs_scale_cols(x):
    return x.apply(lambda col: (col - col.mean()) / col.std() if (len(x) > 1 and col.std() > 0) else col * 0.0)

_scaled = data_all.groupby(['season', 'gameday'], group_keys=False)[ALL_CS + ['elo_diff', 'goalsscored_MOM_player']].apply(_cs_scale_cols)
data_all[ALL_CS] = _scaled[ALL_CS]
data_all['elo_diff'] = _scaled['elo_diff']                                   # --- C_ELO uses this
data_all['goalsscored_MOM_player_sdz'] = _scaled['goalsscored_MOM_player']   # --- F_MOMs uses this
assert data_all[ALL_CS + ['elo_diff', 'goalsscored_MOM_player_sdz']].notna().all().all()

data_all[factors_CS].hist(alpha=0.6, bins=20);

<br>

## 1 &emsp; The Grand Loop

In [ ]:
# ======================================== The Grand Loop ======================================== #
#
# One fold = train through trainT[i], score valT[i] FROZEN. Inside each fold, in order:
#   1) HSGP approximation parameters                 (as in SFM_II__dev)
#   2) the model, built inline                       (SFM_II__dev cells, cfg-branched)
#   3) prior-predictive gate, first fold only        (referee M7.2)
#   4) seeded sampling, numpyro, target_accept 0.9   (referee M7)
#   5) convergence gate                              (referee M7.1)
#   6) frozen scoring: same players via pm.set_data; NEW players via the M2-FIXED extension
#   7) eta parity asserts on BOTH paths              (referee M2 -- the mu-bug guard)
#   8) reduced parameter posterior -> netCDF; ledger append
#
# SMOKE runs the LAST fold only at 200/200 draws -- a pipeline test, not inference.

# --- the settings that produced every number below, ON THE RECORD in every log
# --- (the rung-3 mixup happened because the flag's state was invisible at runtime):
_MODE = ('PRODUCTION' if RUN_PRODUCTION else 'HOLDOUT' if RUN_HOLDOUT else 'VALIDATION-GRID')
print(f'[config] MODE = {_MODE}   (Dev{devVersion} | chains={N_chains} | '
      f'target_accept={TARGET_ACCEPT} | noncenter_zsn={NONCENTER_ZSN} | smoke={SMOKE})')
print(f'[config] folds = {list(zip(dict_EW["trainT"], dict_EW["valT"]))}')
# --- kernel-state tripwire: the flags and the window must agree. They disagree whenever the
# --- USER INTERACTION cell was EDITED but not RE-RUN -- editing a cell does not change the
# --- kernel, and the loop reads the kernel. This catches that silently-wrong run.
if RUN_PRODUCTION and dict_EW.get('valT') != [None]:
    raise AssertionError('RUN_PRODUCTION is True but dict_EW is still a validation window -- '
                         'RE-RUN the USER INTERACTION cell, then this one.')
if (not RUN_PRODUCTION) and dict_EW.get('valT') == [None]:
    raise AssertionError('dict_EW is a production window but RUN_PRODUCTION is False -- '
                         'RE-RUN the USER INTERACTION cell, then this one.')

dict_preds, dict_diags, dict_timing = {}, {}, {}

folds = list(zip(dict_EW['trainT'], dict_EW['valT']))
if SMOKE:
    folds = folds[-1:]
    print('*** SMOKE MODE: last fold only, 200/200 draws -- do NOT export or interpret. ***')

for i_fold, (train_end_f, val_season) in enumerate(folds):

    print('\n' + '=' * 100)
    print(f'==============  Dev{devVersion}  |  train <= {train_end_f}  |  score {val_season} (frozen)  ==============')
    print('=' * 100)

    _t_fold = time.time()
    train_df = data_all[data_all.season <= train_end_f].reset_index(drop=True)
    val_df   = (data_all[data_all.season == val_season].reset_index(drop=True)
                if val_season is not None else train_df.iloc[:0])

    # ---------------------------------- HSGP Parameters ---------------------------------- #
    m_within, c_within = pm.gp.hsgp_approx.approx_hsgp_hyperparams(
        x_range=[0, int(train_df.gameday.max())], lengthscale_range=[5, 25], cov_func="matern52")
    m_long, c_long = pm.gp.hsgp_approx.approx_hsgp_hyperparams(
        x_range=[0, 30], lengthscale_range=[2, 6], cov_func="matern52")
    ls_short_dist  = LS_PRIORS['short']
    ls_medium_dist = LS_PRIORS['medium']
    ls_long_dist   = LS_PRIORS['long']
    ls_era_dist    = LS_PRIORS['era']   # --- era drift is smooth (F_ERA, referee M6)

    # ---------------------------------- Indices & Coords ---------------------------------- #
    players_ordered = train_df["name_player"].sort_values().unique()
    unique_gamedays = np.arange(1, 39)
    unique_seasons  = np.arange(0, 31)

    player_idx  = pd.Categorical(train_df["name_player"], categories=players_ordered).codes
    gameday_idx = pd.Categorical(train_df["gameday"],     categories=unique_gamedays).codes
    assert (player_idx >= 0).all() and (gameday_idx >= 0).all()

    timescale = ['short', 'medium', 'long'] if cfg['hsgp'] == 'both' else ['long']
    COORDS = {
        "event": np.array([0, 1, 2, 3]),
        "factor_team": factors_team,
        "factor_player": factors_player,
        "gameday": unique_gamedays,
        "obs_id": train_df.index,
        "player": players_ordered,
        "season": unique_seasons,
        "cal_season": np.arange(0, 31),
        "league": leagues_ordered,
        "timescale": timescale,
    }

    # --------------- Priors for the Cutpoints of the Ordered-Logistic (this fold's train) --------------- #
    empirical_probs = train_df["goals_cats"].value_counts(normalize=True).sort_index().to_numpy()
    cutpoints_mu_standard = norm.ppf(empirical_probs.cumsum()[:-1])
    delta_prior = np.diff(cutpoints_mu_standard)

    # ---------------------------------- Set the Model (as SFM_II__dev) ---------------------------------- #
    with pm.Model(coords=COORDS) as SFM_II__ew:

        # --- Data containers:
        factor_data__team = pm.Data("factor_data__team", train_df[factors_team].to_numpy(),
                                    dims=("obs_id", "factor_team"))
        if cfg['mom']:
            factor_data__player = pm.Data("factor_data__player", train_df[factors_player].to_numpy(),
                                          dims=("obs_id", "factor_player"))
        gameday_id = pm.Data("gameday_id", gameday_idx, dims="obs_id")
        player_id = pm.Data("player_id", player_idx, dims="obs_id")
        season_id = pm.Data("season_id", train_df["season_nbr"].to_numpy(), dims="obs_id")
        if cfg['era']:
            cal_season_id = pm.Data("cal_season_id", train_df["cal_season_nbr"].to_numpy(), dims="obs_id")
        if cfg['league']:
            league_id = pm.Data("league_id", train_df["league_nbr"].to_numpy(), dims="obs_id")
        goals_obs = pm.Data("goals_obs", train_df["goals_cats"].to_numpy(), dims="obs_id")

        # ---------------------------------- Player's Baseline Skill ---------------------------------- #
        # --- Baseline Skill + Zero-Sum Normal (exactly as SFM_II__dev):
        intercept_sigma = 5
        sd = pm.Exponential("player_effect_diversity", 1)
        baseline_sigma = pt.sqrt(intercept_sigma**2 + sd**2 / len(COORDS["player"]))
        baseline = baseline_sigma * pm.Normal("baseline")
        if NONCENTER_ZSN:
            # --- rung 3: unit-scale ZSN, sd applied outside -- same posterior, kinder geometry
            player_effect = pm.Deterministic(
                "player_effect",
                baseline + sd * pm.ZeroSumNormal("player_effect_raw", sigma=1.0, dims="player"),
                dims="player")
        else:
            player_effect = pm.Deterministic(
                "player_effect",
                baseline + pm.ZeroSumNormal("player_effect_raw", sigma=sd, dims="player"),
                dims="player")

        # ---------------------------------- Maturity Effects (via HSGPs) ---------------------------------- #
        X_gamedays = pm.Data("X_gamedays", unique_gamedays, dims="gameday")[:, None]
        X_seasons = pm.Data("X_seasons", unique_seasons, dims="season")[:, None]

        ## 1% chance that amplitude > 2 goals
        alpha_scale, upper_scale = 0.01, 2.0

        if cfg['hsgp'] == 'both':
            amplitude = pm.Exponential("amplitude", lam=-np.log(alpha_scale) / upper_scale, dims="timescale")
            ls = pm.InverseGamma(
                "ls",
                alpha=np.array([ls_short_dist.alpha, ls_medium_dist.alpha, ls_long_dist.alpha]),
                beta=np.array([ls_short_dist.beta, ls_medium_dist.beta, ls_long_dist.beta]),
                dims="timescale")
            cov_short = amplitude[0] ** 2 * pm.gp.cov.Matern52(input_dim=1, ls=ls[0])
            cov_medium = amplitude[1] ** 2 * pm.gp.cov.Matern52(input_dim=1, ls=ls[1])
            cov_within = cov_short + cov_medium
            cov_long = amplitude[2] ** 2 * pm.gp.cov.Matern52(input_dim=1, ls=ls[2])

            gp_within = pm.gp.HSGP(m=[m_within], c=c_within, cov_func=cov_within)
            basis_vectors_within, sqrt_psd_within = gp_within.prior_linearized(X=X_gamedays)
            basis_coeffs_within = pm.Normal("basis_coeffs_within", shape=gp_within.n_basis_vectors)
            f_within = pm.Deterministic("f_within",
                basis_vectors_within @ (basis_coeffs_within * sqrt_psd_within), dims="gameday")
        elif cfg['hsgp'] == 'long':
            amplitude = pm.Exponential("amplitude", lam=-np.log(alpha_scale) / upper_scale, shape=1)
            ls = pm.InverseGamma("ls", alpha=np.array(ls_long_dist.alpha),
                                 beta=np.array(ls_long_dist.beta), shape=1)
            cov_long = amplitude ** 2 * pm.gp.cov.Matern52(input_dim=1, ls=ls)

        # --- hsgp == 'none' (A-family): no GPs at all
        if cfg['hsgp'] != 'none':
            gp_long = pm.gp.HSGP(m=[m_long], c=c_long, cov_func=cov_long)
            basis_vectors_long, sqrt_psd_long = gp_long.prior_linearized(X=X_seasons)
            basis_coeffs_long = pm.Normal("basis_coeffs_long", shape=gp_long.n_basis_vectors)
            f_long = pm.Deterministic("f_long",
                basis_vectors_long @ (basis_coeffs_long * sqrt_psd_long), dims="season")

        # ------------------------ Calendar-Era Effect (F_ERA; referee M6) ------------------------ #
        # --- Shared HSGP over calendar season (fixed 0..30 grid) -- absorbs the secular scoring
        # --- drift (~18% over the window) so it stops loading onto player effects. At score time
        # --- the frozen season sits one step beyond training; the mean-reverting HSGP extrapolates.
        if cfg['era']:
            X_cal = pm.Data("X_cal_seasons", np.arange(0, 31), dims="cal_season")[:, None]
            amplitude_era = pm.Exponential("amplitude_era", lam=-np.log(alpha_scale) / upper_scale)
            ls_era = pm.InverseGamma("ls_era", alpha=ls_era_dist.alpha, beta=ls_era_dist.beta)
            cov_era = amplitude_era ** 2 * pm.gp.cov.Matern52(input_dim=1, ls=ls_era)
            gp_era = pm.gp.HSGP(m=[m_long], c=c_long, cov_func=cov_era)
            basis_vectors_era, sqrt_psd_era = gp_era.prior_linearized(X=X_cal)
            basis_coeffs_era = pm.Normal("basis_coeffs_era", shape=gp_era.n_basis_vectors)
            f_era = pm.Deterministic("f_era",
                basis_vectors_era @ (basis_coeffs_era * sqrt_psd_era), dims="cal_season")
            era_term = f_era[cal_season_id]
        else:
            era_term = 0.0

        # ------------------------ League Intercepts (F_LEAGUE) ------------------------ #
        # --- Zero-sum so league effects cannot compete with the player baseline for the
        # --- overall level (only RELATIVE league differences are identified). sigma = 0.1 is
        # --- weakly informative on the logit scale: the observed big-5 spread is
        # --- 0.195-0.205 goals/appearance, i.e. ~0.06 in logits, so +-0.2 at 2sd is generous.
        if cfg['league']:
            league_effect = pm.ZeroSumNormal("league_effect", sigma=0.1, dims="league")
            league_term = league_effect[league_id]
        else:
            league_term = 0.0

        # ---------------------------------- alpha ---------------------------------- #
        if cfg['hsgp'] == 'both':
            alpha = pm.Deterministic("alpha",
                player_effect[player_id] + f_within[gameday_id] + f_long[season_id]
                + era_term + league_term,
                dims="obs_id")
        elif cfg['hsgp'] == 'long':
            alpha = pm.Deterministic("alpha",
                player_effect[player_id] + f_long[season_id] + era_term + league_term, dims="obs_id")
        else:
            alpha = pm.Deterministic("alpha",
                player_effect[player_id] + era_term + league_term, dims="obs_id")

        # ---------------------------------- The Team-Factors ---------------------------------- #
        beta__team = pm.Normal("beta__team", sigma=2.5, dims="factor_team")
        if cfg['mom']:
            # --- pooled ONLY: the per-player betas were rejected by LOO (referee M1.3)
            beta__player = pm.Normal("beta__player", sigma=2.5, dims="factor_player")
            eta = pm.Deterministic("eta",
                alpha + (factor_data__player * beta__player).sum(axis=-1)
                + pm.math.dot(factor_data__team, beta__team), dims="obs_id")
        else:
            eta = pm.Deterministic("eta",
                alpha + pm.math.dot(factor_data__team, beta__team), dims="obs_id")

        # ---------------------------------- Likelihood (as SFM_II__dev) ---------------------------------- #
        cutpoint_offset = 4
        delta_mean = pm.Normal("delta_mean", mu=delta_prior * cutpoint_offset, sigma=1, shape=2)
        delta_sigma = pm.Exponential("delta_sigma", 1, shape=2)
        delta_player = delta_mean + delta_sigma * pm.Normal("delta_player", shape=(len(COORDS["player"]), 2))
        cutpoints = pm.Deterministic("cutpoints",
            pt.concatenate([pt.full((player_effect.shape[0], 1), cutpoint_offset),
                            pt.cumsum(pt.softplus(delta_player), axis=-1) + cutpoint_offset], axis=-1))
        # --- index with the pm.Data container (player_id), NOT the static codes -- set_data
        # --- must re-route the cutpoints for the frozen-season scoring:
        pm.OrderedLogistic("goals_scored", cutpoints=cutpoints[player_id], eta=eta,
                           observed=goals_obs, dims="obs_id")

    # ---------------------------------- Prior-Predictive Gate (M7.2) ---------------------------------- #
    if i_fold == 0:
        with SFM_II__ew:
            idata_prior = pm.sample_prior_predictive(draws=100, var_names=["goals_scored"], random_seed=seed)
        _pc = np.bincount(idata_prior.prior_predictive["goals_scored"].values.ravel(), minlength=4)
        _ec = np.bincount(train_df['goals_cats'], minlength=4) / len(train_df)
        print(f'[prior pred] shares {np.round(_pc / _pc.sum(), 4)}  vs empirical {np.round(_ec, 4)}')
        del idata_prior

    # ---------------------------------- Inference! ---------------------------------- #
    # ---------------------------------- Memory (GPU OOM guard) ---------------------------------- #
    # --- Store FREE RVs ONLY. The obs-sized Deterministics are enormous at full draws:
    # ---   alpha, eta           2 x 1000 x 314k x 8B  ~  5 GB each
    # ---   goals_scored_probs   2 x 1000 x 314k x 4   ~ 20 GB
    # ---   cutpoints[player_id] 2 x 1000 x 314k x 3   ~ 15 GB   <- this one OOM'd the A100
    # --- and we need NONE of them from the training posterior: sample_posterior_predictive
    # --- recomputes them from the free RVs for the ~20k FROZEN rows during scoring, where
    # --- they cost ~0.3 GB. (SMOKE survived only because 200 draws is 1/5 the size.)
    # --- Storage-only change: the posterior itself is identical.
    _free_rvs = [rv.name for rv in SFM_II__ew.free_RVs]

    _sample_kwargs = dict(nuts_sampler="numpyro",
                          target_accept=TARGET_ACCEPT,           # --- M7 + divergence ladder (USER INTERACTION)
                          chains=N_chains, cores=1,
                          draws=200 if SMOKE else 1000,
                          tune=200 if SMOKE else 1000,
                          random_seed=seed,                      # --- M7.3: reproducible
                          # --- postprocessing on the host: belt-and-braces if var_names is ignored
                          nuts_sampler_kwargs={"chain_method": "vectorized",
                                               "postprocessing_backend": "cpu"})

    _t_sample0 = time.time()
    with SFM_II__ew:
        try:
            idata = pm.sample(var_names=_free_rvs, **_sample_kwargs)
        except TypeError:
            print('[memory] this PyMC ignores var_names on the numpyro path -- '
                  'falling back to CPU postprocessing only')
            idata = pm.sample(**_sample_kwargs)

    print(f'[memory] stored {len(idata.posterior.data_vars)} posterior variables '
          f'({sum(v.nbytes for v in idata.posterior.data_vars.values())/1e6:.0f} MB); '
          f'obs-sized deterministics recomputed at scoring time')

    _t_sample = time.time() - _t_sample0
    print(f'[timing] sampling {_t_sample/60:.1f} min '
          f'({_t_sample/(2*(200 if SMOKE else 1000)):.2f} s/it)')

    # ---------------------------------- Convergence Gate (M7.1) ---------------------------------- #
    # --- The CHECK is non-negotiable (M7.1: every historical SFM number rested on unexamined
    # --- chains). Summarising all ~8.7k coordinates is not: 'delta_player' (n_players x 2)
    # --- and 'player_effect_raw' (n_players) dominate the cost while carrying almost no
    # --- extra information -- systematic non-convergence (a funnel in the player scale) does
    # --- not hide in a random 250 of 2,897 coordinates. So: structural parameters in FULL,
    # --- the two big hierarchical arrays on a seeded subsample, r_hat + ess_bulk only
    # --- (az.summary also computes ess_tail and two MCSEs the gate never reads).
    # --- Deterministics are excluded: the cutpoints' first column is the CONSTANT 4 by
    # --- construction -> zero within-chain variance -> 0/0 warnings. The free RVs below
    # --- fully determine every Deterministic. DIAGNOSTICS ONLY -- the posterior is untouched.
    _DETS = {'player_effect', 'f_within', 'f_long', 'f_era', 'cutpoints', 'alpha', 'eta'}
    _BIG = {'player_effect_raw', 'delta_player'}
    _DIAG_SUB = 250

    _post = idata.posterior
    _rng_diag = np.random.default_rng(seed)
    _checked, _n_coord, _n_total = {}, 0, 0
    for _v in _post.data_vars:
        if "obs_id" in _post[_v].dims or _v in _DETS:
            continue
        _da = _post[_v]
        _n_total += int(np.prod([_da.sizes[d] for d in _da.dims if d not in ('chain', 'draw')]) or 1)
        if _v in _BIG:
            _pdim = [d for d in _da.dims if d not in ('chain', 'draw')][0]
            _n = _da.sizes[_pdim]
            if _n > _DIAG_SUB:
                _da = _da.isel({_pdim: np.sort(_rng_diag.choice(_n, _DIAG_SUB, replace=False))})
        _checked[_v] = _da
        _n_coord += int(np.prod([_da.sizes[d] for d in _da.dims if d not in ('chain', 'draw')]) or 1)

    # --- az.rhat/az.ess return a DATASET even for a single DataArray -> reduce via to_array()
    _rh = {v: float(az.rhat(da).to_array().max()) for v, da in _checked.items()}
    _es = {v: float(az.ess(da).to_array().min()) for v, da in _checked.items()}
    _max_rhat, _min_ess = max(_rh.values()), min(_es.values())
    _worst_rhat = max(_rh, key=_rh.get)
    _worst_ess = min(_es, key=_es.get)

    _ndiv = int(idata.sample_stats["diverging"].sum().values) if "diverging" in idata.sample_stats else -1
    _ok = (_max_rhat <= 1.01) and (_min_ess >= 100 * N_chains) and (_ndiv == 0)
    _verdict = 'OK' if _ok else ('not converged (EXPECTED in SMOKE -- 200/200 draws)' if SMOKE
                                 else '!! CHECK CONVERGENCE')
    print(f'[diagnostics] divergences={_ndiv}  max_rhat={_max_rhat:.4f} ({_worst_rhat})  '
          f'min_ess_bulk={_min_ess:.0f} ({_worst_ess})  ->  {_verdict}')
    # --- no silent caps: state what was actually checked
    print(f'[diagnostics] {_n_coord:,} of {_n_total:,} coordinates checked '
          f'(hierarchical arrays subsampled to {_DIAG_SUB})')
    if _ndiv > 0:
        # --- WHERE do divergent draws sit? For each checked variable: max |z| of the
        # --- divergent-draw mean vs the rest. The largest |z| marks the neck -- that is
        # --- what rung 3 (or its successor) should target.
        _div_mask = idata.sample_stats["diverging"].values.reshape(-1).astype(bool)
        if 0 < _div_mask.sum() < _div_mask.size:
            _loc = {}
            for _v, _da in _checked.items():
                _flat = _da.transpose('chain', 'draw', ...).values.reshape(_div_mask.size, -1)
                _sd_o = _flat[~_div_mask].std(axis=0) + 1e-12
                _loc[_v] = float(np.max(np.abs(_flat[_div_mask].mean(axis=0)
                                               - _flat[~_div_mask].mean(axis=0)) / _sd_o))
            print('   divergence location (max |z|, divergent vs rest):',
                  {k: round(v, 2) for k, v in sorted(_loc.items(), key=lambda kv: -kv[1])[:6]})
    if not _ok and not SMOKE:
        print('   worst r_hat by variable:',
              {k: round(v, 4) for k, v in sorted(_rh.items(), key=lambda kv: -kv[1])[:6]})
        print('   worst ess_bulk by variable:',
              {k: round(v) for k, v in sorted(_es.items(), key=lambda kv: kv[1])[:6]})
    dict_diags[val_season] = dict(divergences=_ndiv, max_rhat=_max_rhat, min_ess_bulk=_min_ess,
                                  n_chains=N_chains, target_accept=TARGET_ACCEPT,
                                  noncenter_zsn=NONCENTER_ZSN,
                                  worst_rhat_var=_worst_rhat, worst_ess_var=_worst_ess,
                                  coords_checked=_n_coord, coords_total=_n_total,
                                  converged_ok=bool(_ok))

    # --------------------------- PRODUCTION: fit only, nothing to score --------------------------- #
    if RUN_PRODUCTION:
        prod = dict(model=SFM_II__ew, idata=idata, train_df=train_df,
                    players_ordered=players_ordered, player_idx=player_idx, gameday_idx=gameday_idx,
                    unique_seasons=unique_seasons, unique_gamedays=unique_gamedays,
                    m_within=m_within, c_within=c_within, m_long=m_long, c_long=c_long,
                    ls_short=ls_short_dist, ls_medium=ls_medium_dist, ls_long=ls_long_dist)
        dict_timing[train_end_f] = dict(sampling_min=_t_sample / 60,
                                        fold_min=(time.time() - _t_fold) / 60)
        print(f'[production] fit complete on {len(train_df):,} rows through {train_end_f} '
              f'({len(players_ordered):,} players). Nothing scored -- run "Export -- Production Model".')
        continue

    # ================================ Score the FROZEN Season ================================ #
    is_new = ~val_df['name_player'].isin(players_ordered)
    data_idx__same = val_df.index[~is_new]
    data_idx__new  = val_df.index[is_new]
    # --- both are free RVs, so they survive the var_names filter -- assert rather than assume
    assert 'beta__team' in idata.posterior, 'beta__team missing from the stored posterior'
    if cfg['mom']:
        assert 'beta__player' in idata.posterior, 'beta__player missing from the stored posterior'
    _bt = idata.posterior['beta__team']
    _out_frames = []

    # ---------------------------------- SAME Players: pm.set_data ---------------------------------- #
    SFM_II__sameP = copy.deepcopy(SFM_II__ew)
    gameday_idx__same = pd.Categorical(val_df.loc[data_idx__same, 'gameday'], categories=unique_gamedays).codes
    player_idx__same = pd.Categorical(val_df.loc[data_idx__same, 'name_player'], categories=players_ordered).codes
    assert (gameday_idx__same >= 0).all() and (player_idx__same >= 0).all()

    with SFM_II__sameP:
        new_data = {"season_id": val_df.loc[data_idx__same, "season_nbr"].values,
                    "gameday_id": gameday_idx__same,
                    "player_id": player_idx__same,
                    "factor_data__team": val_df.loc[data_idx__same, factors_team].values,
                    # --- Just a placeholder:
                    "goals_obs": np.zeros(len(data_idx__same), dtype=np.int64)}
        if cfg['mom']:
            new_data["factor_data__player"] = val_df.loc[data_idx__same, factors_player].values
        if cfg['era']:
            new_data["cal_season_id"] = val_df.loc[data_idx__same, "cal_season_nbr"].values
        if cfg['league']:
            new_data["league_id"] = val_df.loc[data_idx__same, "league_nbr"].values
        pm.set_data(coords={"obs_id": data_idx__same}, new_data=new_data)

        oos_preds__same = pm.sample_posterior_predictive(
            idata, var_names=["alpha", "eta", "goals_scored_probs"], predictions=True,
            compile_kwargs={"mode": "NUMBA"}, random_seed=seed)

    # --- M3.1: the predictive is the posterior MEAN over draws:
    probs__same = oos_preds__same['predictions']['goals_scored_probs'].mean(('chain', 'draw')).to_numpy()

    # --- eta Parity (referee M2 -- the mu-bug guard), same players:
    _recon = (oos_preds__same['predictions']['alpha']
              + xr.DataArray(val_df.loc[data_idx__same, factors_team].to_numpy(),
                             dims=('obs_id', 'factor_team')) @ _bt)
    if cfg['mom']:
        _recon = _recon + xr.DataArray(val_df.loc[data_idx__same, factors_player].to_numpy(),
                                       dims=('obs_id', 'factor_player')) @ idata.posterior['beta__player']
    _dev_same = float(np.abs(_recon - oos_preds__same['predictions']['eta']).max())
    assert _dev_same < 1e-6, f'ETA PARITY FAILED (same players): {_dev_same:.3e}'

    _df_s = val_df.loc[data_idx__same, ['name_player', 'id_match', 'season', 'gameday', 'goals_cats']].copy()
    _df_s['is_new'] = False
    for _k in range(4):
        _df_s[f'p{_k}'] = probs__same[:, _k]
    _out_frames.append(_df_s)
    del SFM_II__sameP, oos_preds__same

    # ---------------------------------- NEW Players (M2-FIXED path) ---------------------------------- #
    if len(data_idx__new) > 0:
        players_ordered__new = val_df.loc[data_idx__new, 'name_player'].sort_values().unique()
        player_idx__new = pd.Categorical(val_df.loc[data_idx__new, 'name_player'], categories=players_ordered__new).codes
        gameday_idx__new = pd.Categorical(val_df.loc[data_idx__new, 'gameday'], categories=unique_gamedays).codes

        SFM_II__newP = copy.deepcopy(SFM_II__ew)
        with SFM_II__newP:
            SFM_II__newP.add_coord("player__new", players_ordered__new)
            SFM_II__newP.add_coord("obs_id__new", data_idx__new)
            SFM_II__newP.add_coord("season__new", unique_seasons)

            X_seasons__new = pm.Data("X_seasons__new", unique_seasons, dims="season__new")[:, None]
            factor_data__new_team = pm.Data("factor_data__new_team",
                val_df.loc[data_idx__new, factors_team].to_numpy(), dims=("obs_id__new", "factor_team"))
            if cfg['mom']:
                factor_data__new_player = pm.Data("factor_data__new_player",
                    val_df.loc[data_idx__new, factors_player].to_numpy(), dims=("obs_id__new", "factor_player"))
            gameday_id__new = pm.Data("gameday_id__new", gameday_idx__new, dims="obs_id__new")
            player_id__new = pm.Data("player_id__new", player_idx__new, dims="obs_id__new")
            season_id__new = pm.Data("season_id__new", val_df.loc[data_idx__new, "season_nbr"].values, dims="obs_id__new")
            if cfg['era']:
                cal_season_id__new = pm.Data("cal_season_id__new",
                    val_df.loc[data_idx__new, "cal_season_nbr"].values, dims="obs_id__new")
            if cfg['league']:
                league_id__new = pm.Data("league_id__new",
                    val_df.loc[data_idx__new, "league_nbr"].values, dims="obs_id__new")
            goals_obs__new = pm.Data("goals_obs__new", np.zeros(len(data_idx__new), dtype=np.int64), dims="obs_id__new")

            # --- Baseline Skill (using fitted parameters)
            # --- M2.1 FIX: the trained baseline is baseline_sigma * Normal("baseline") -- the raw
            # --- node alone shifted every new player's eta by ~ -4z (a ~6x under-prediction).
            # --- The multiplier must match TRAINING exactly: N = len(players_ordered).
            intercept_sigma = 5
            sd = SFM_II__newP.player_effect_diversity
            baseline_sigma = pt.sqrt(intercept_sigma ** 2 + sd ** 2 / len(players_ordered))
            baseline = baseline_sigma * SFM_II__newP.baseline

            if NONCENTER_ZSN:
                player_effect__new = pm.Deterministic("player_effect__new",
                    baseline + sd * pm.ZeroSumNormal("player_effect_raw__new", sigma=1.0, dims="player__new"),
                    dims="player__new")
            else:
                player_effect__new = pm.Deterministic("player_effect__new",
                    baseline + pm.ZeroSumNormal("player_effect_raw__new", sigma=sd, dims="player__new"),
                    dims="player__new")

            if cfg['hsgp'] != 'none':
                # --- GPs: Cross-Seasonal Variation (reconstruct on the same fixed grid, fitted params):
                _amp, _ls = SFM_II__newP.amplitude, SFM_II__newP.ls
                if cfg['hsgp'] == 'both':
                    cov_long__new = _amp[2] ** 2 * pm.gp.cov.Matern52(input_dim=1, ls=_ls[2])
                else:
                    cov_long__new = _amp ** 2 * pm.gp.cov.Matern52(input_dim=1, ls=_ls)
                gp_long__new = pm.gp.HSGP(m=[m_long], c=c_long, cov_func=cov_long__new)
                basis_vectors_long__new, sqrt_psd_long__new = gp_long__new.prior_linearized(X=X_seasons__new)
                f_long__new = pm.Deterministic("f_long__new",
                    basis_vectors_long__new @ (SFM_II__newP.basis_coeffs_long * sqrt_psd_long__new),
                    dims="season__new")

            # --- era term: f_era lives on the fixed 0..30 grid -> reuse the fitted deterministic
            era_term__new = SFM_II__newP.f_era[cal_season_id__new] if cfg['era'] else 0.0
            # --- leagues are a fixed, fully-observed set -> reuse the FITTED zero-sum effects
            league_term__new = SFM_II__newP.league_effect[league_id__new] if cfg['league'] else 0.0

            # --- Player's Alpha:
            if cfg['hsgp'] == 'both':
                alpha__new = pm.Deterministic("alpha__new",
                    player_effect__new[player_id__new] + SFM_II__newP.f_within[gameday_id__new]
                    + f_long__new[season_id__new] + era_term__new + league_term__new,
                    dims="obs_id__new")
            elif cfg['hsgp'] == 'long':
                alpha__new = pm.Deterministic("alpha__new",
                    player_effect__new[player_id__new] + f_long__new[season_id__new]
                    + era_term__new + league_term__new,
                    dims="obs_id__new")
            else:
                alpha__new = pm.Deterministic("alpha__new",
                    player_effect__new[player_id__new] + era_term__new + league_term__new,
                    dims="obs_id__new")

            # --- Team Factors (use fitted slope from training):
            beta__team_new = SFM_II__newP.beta__team
            if cfg['mom']:
                # --- M2.2 FIX: the pooled beta__player is a GLOBAL fitted coefficient -- reuse the
                # --- trained node (redrawing it under a new name sampled it from the PRIOR).
                beta__player_new = SFM_II__newP.beta__player
                eta__new = pm.Deterministic("eta__new",
                    alpha__new + (factor_data__new_player * beta__player_new).sum(axis=-1)
                    + pm.math.dot(factor_data__new_team, beta__team_new), dims="obs_id__new")
            else:
                eta__new = pm.Deterministic("eta__new",
                    alpha__new + pm.math.dot(factor_data__new_team, beta__team_new), dims="obs_id__new")

            # --- New players draw cutpoint offsets from the posterior hyperpriors (as SFM_II__dev):
            cutpoint_offset = 4
            delta_player__new = SFM_II__newP.delta_mean + SFM_II__newP.delta_sigma * pm.Normal(
                "delta_player__new", shape=(len(players_ordered__new), 2))
            cutpoints__new = pm.Deterministic("cutpoints__new",
                pt.concatenate([pt.full((player_effect__new.shape[0], 1), cutpoint_offset),
                                pt.cumsum(pt.softplus(delta_player__new), axis=-1) + cutpoint_offset], axis=-1))
            pm.OrderedLogistic("goals_scored__new", cutpoints=cutpoints__new[player_id__new],
                               eta=eta__new, observed=goals_obs__new, dims="obs_id__new")

            oos_preds__new = pm.sample_posterior_predictive(
                idata, var_names=["alpha__new", "eta__new", "goals_scored__new_probs"],
                predictions=True, compile_kwargs={"mode": "NUMBA"}, random_seed=seed)

        probs__new = oos_preds__new['predictions']['goals_scored__new_probs'].mean(('chain', 'draw')).to_numpy()

        # --- eta Parity, new players (the path the revision repaired):
        _recon_n = (oos_preds__new['predictions']['alpha__new']
                    + xr.DataArray(val_df.loc[data_idx__new, factors_team].to_numpy(),
                                   dims=('obs_id__new', 'factor_team')) @ _bt)
        if cfg['mom']:
            _recon_n = _recon_n + xr.DataArray(val_df.loc[data_idx__new, factors_player].to_numpy(),
                                               dims=('obs_id__new', 'factor_player')) @ idata.posterior['beta__player']
        _dev_new = float(np.abs(_recon_n - oos_preds__new['predictions']['eta__new']).max())
        assert _dev_new < 1e-6, f'ETA PARITY FAILED (new players): {_dev_new:.3e}'

        _df_n = val_df.loc[data_idx__new, ['name_player', 'id_match', 'season', 'gameday', 'goals_cats']].copy()
        _df_n['is_new'] = True
        for _k in range(4):
            _df_n[f'p{_k}'] = probs__new[:, _k]
        _out_frames.append(_df_n)
        del SFM_II__newP, oos_preds__new

    # ---------------------------------- Ledger & Housekeeping ---------------------------------- #
    res = pd.concat(_out_frames)
    res.insert(0, 'variant', devVersion)
    res.insert(1, 'val_season', val_season)
    dict_preds[val_season] = res
    print(f'[eta parity] PASS  |  scored {len(res)} rows ({int(res.is_new.sum())} new-player rows)')

    # --- M7.4: reduced parameter posterior (obs-sized deterministics dropped; a few MB):
    if not SMOKE:
        _tag = '_HOLDOUT' if RUN_HOLDOUT else ''
        az.InferenceData(posterior=idata.posterior.drop_dims('obs_id', errors='ignore'),
                         sample_stats=idata.sample_stats).to_netcdf(
            f"{directory}/10_data/102_Development/Posterior__SFM_II_EW_Dev{devVersion}_val{val_season.replace('/', '')}{_tag}.nc")

    dict_timing[val_season] = dict(sampling_min=_t_sample / 60,
                                   fold_min=(time.time() - _t_fold) / 60)
    print(f'[timing] fold total {dict_timing[val_season]["fold_min"]:.1f} min')

    del SFM_II__ew, idata
    gc.collect()

# --- Grid budget: extrapolate what the remaining variants will cost at THIS machine's speed
_mean_fold = np.mean([v['fold_min'] for v in dict_timing.values()]) if dict_timing else float('nan')
_scale = (1000 / 200) if SMOKE else 1.0     # smoke folds run 1/5 of the iterations
_n_fits = len(VARIANTS) * len(list(zip(dict_EW['trainT'], dict_EW['valT'])))
print(f'\nGrand Loop complete.  mean fold {_mean_fold:.1f} min'
      + (f'  ->  ~{_mean_fold * _scale:.0f} min/fold at full draws' if SMOKE else ''))
print(f'  projected FULL grid ({len(VARIANTS)} variants x {len(dict_EW["valT"])} folds = {_n_fits} fits): '
      f'{_mean_fold * _scale * _n_fits / 60:.1f} GPU-hours')

<br>

## 2 &emsp; Evaluation — OOS

In [ ]:
# ================================== Out-of-Sample Evaluation ================================== #
#
# ONE evaluation cell (referee M7.2), reporting overall AND by gameday bucket AND new/same
# players. Probabilities are posterior MEANS (M3.1); the naive is the MARGINAL-FREQUENCY
# forecast (M3.3) -- the degenerate P(0)=1 predictor and its epsilon-driven logLik are gone.

assert not RUN_PRODUCTION, 'PRODUCTION run: nothing was scored -- skip to "Export -- Production Model".'

perrow = pd.concat(dict_preds.values(), ignore_index=True)
_P = perrow[['p0', 'p1', 'p2', 'p3']].to_numpy()
_y = perrow['goals_cats'].to_numpy()
perrow['logloss'] = perrow_logloss(_P, _y)
perrow['rps'] = perrow_rps(_P, _y)

print(f'Version: Dev{devVersion}   (probability rows sum to {_P.sum(axis=1).mean():.4f} on average)')

# --- per frozen season, then overall:
_rows = [dict(split=s, **summarize_probs(dict_preds[s][['p0','p1','p2','p3']].to_numpy(),
                                         dict_preds[s]['goals_cats'].to_numpy())) for s in dict_preds]
_rows.append(dict(split='all', **summarize_probs(_P, _y)))
df_eval = pd.DataFrame(_rows).set_index('split')
print(df_eval.round(4).to_string())

# --- marginal-frequency naive on the SAME rows (per fold, train-window frequencies):
_nrows = []
for _tt, _vv in zip(dict_EW['trainT'], dict_EW['valT']):
    if _vv not in dict_preds:
        continue
    _marg = data_all.loc[data_all.season <= _tt, 'goals_cats'].value_counts(normalize=True).sort_index().to_numpy()
    _yv = dict_preds[_vv]['goals_cats'].to_numpy()
    _nrows.append(dict(split=_vv, **summarize_probs(np.tile(_marg, (len(_yv), 1)), _yv)))
print('\n--- marginal-frequency naive (M3.3) ---')
print(pd.DataFrame(_nrows).set_index('split').round(4).to_string())

# --- BY GAMEDAY BUCKET (the SFMMO-M1 guard: no regime may silently vanish from this table):
BUCKETS = [(1, 2, 'gd 1-2  (cold start)'), (3, 10, 'gd 3-10'), (11, 20, 'gd 11-20'), (21, 99, 'gd 21+')]
_brows = []
for _lo, _hi, _lbl in BUCKETS:
    _m = (perrow['gameday'] >= _lo) & (perrow['gameday'] <= _hi)
    if _m.sum() == 0:
        _brows.append(dict(bucket=_lbl, n=0))
        print(f'  !! bucket {_lbl!r} is EMPTY -- the pipeline is dropping this regime.')
        continue
    _brows.append(dict(bucket=_lbl, **summarize_probs(perrow.loc[_m, ['p0','p1','p2','p3']].to_numpy(),
                                                      perrow.loc[_m, 'goals_cats'].to_numpy())))
df_bucket = pd.DataFrame(_brows).set_index('bucket')
print('\n--- by gameday bucket ---')
print(df_bucket.round(4).to_string())

# --- NEW vs SAME players (the regime repaired by M2 -- watch it explicitly):
_srows = []
for _flag, _lbl in [(False, 'same players'), (True, 'new players')]:
    _m = perrow['is_new'] == _flag
    if _m.sum() > 0:
        _srows.append(dict(split=_lbl, **summarize_probs(perrow.loc[_m, ['p0','p1','p2','p3']].to_numpy(),
                                                         perrow.loc[_m, 'goals_cats'].to_numpy())))
print('\n--- new vs same players (M2 watch) ---')
print(pd.DataFrame(_srows).set_index('split').round(4).to_string())

print('\n--- sampling diagnostics by fold ---')
print(pd.DataFrame(dict_diags).T.to_string())

In [ ]:
# ================================== Export ================================== #

assert not RUN_PRODUCTION, 'PRODUCTION run: use the "Export -- Production Model" cell below.'
assert not SMOKE, 'SMOKE run -- artifacts from 200/200 draws must not be exported. Set SMOKE = False and rerun.'

import pickle
import cloudpickle

_tag = '__EW_HOLDOUT' if RUN_HOLDOUT else '__EW'
pickle_filepath = f'{directory}/10_data/102_Development/Evaluation__SFM_II_Dev{devVersion}__scaleCS{_tag}.pkl'
dict_to_save = {'devVersion': devVersion,
                'cfg': cfg,
                'dict_EW': dict_EW,
                'perrow': perrow,
                'df_eval': df_eval,
                'df_bucket': df_bucket,
                'diagnostics': pd.DataFrame(dict_diags).T,
                'seed': seed}

with open(pickle_filepath, 'wb') as f:
    cloudpickle.dump(dict_to_save, f)

print(f'Fine. Version: Dev{devVersion}  ->  {pickle_filepath}')

<br>

## 3 &emsp; Cross-Variant Verdict (pre-registered)

Run this after two or more variants have been exported. The rule was fixed on 2026-08-04,
**before any run**: a challenger replaces the incumbent only if it wins **both** log-loss and
RPS with paired |t| ≥ 2.5 on identical player-appearances pooled across folds; ties resolve
to parsimony (fewer factors, then fewer HSGP timescales). Holdout exports are excluded.

In [ ]:
# ============================ Cross-Variant Verdict (pre-registered) ============================ #

_files = sorted(glob.glob(f'{directory}/10_data/102_Development/Evaluation__SFM_II_Dev*__scaleCS__EW.pkl'))
_led = {}
for _fp in _files:
    _d = pd.read_pickle(_fp)
    _led[_d['devVersion']] = _d['perrow']
print('variants on disk:', sorted(_led))

if INCUMBENT not in _led:
    print(f'!! incumbent Dev{INCUMBENT} not exported yet -- run it first.')
else:
    _key = ['val_season', 'name_player', 'id_match']
    _base = _led[INCUMBENT].set_index(_key)
    print(f'incumbent: Dev{INCUMBENT}  ({len(_base)} rows)\n')
    for _v in sorted(_led):
        if _v == INCUMBENT:
            continue
        _ch = _led[_v].set_index(_key)
        _common = _base.index.intersection(_ch.index)
        _dll = _ch.loc[_common, 'logloss'] - _base.loc[_common, 'logloss']
        _drp = _ch.loc[_common, 'rps'] - _base.loc[_common, 'rps']
        _t_ll, _t_rp = paired_t(_dll), paired_t(_drp)
        _wins = (_dll.mean() < 0) and (_drp.mean() < 0) and (abs(_t_ll) >= 2.5) and (abs(_t_rp) >= 2.5)
        print(f'Dev{_v:7s} vs Dev{INCUMBENT}:  d_logloss {_dll.mean():+.5f} (t {_t_ll:+.2f})   '
              f'd_rps {_drp.mean():+.5f} (t {_t_rp:+.2f})   ->  '
              f'{"QUALIFIES" if _wins else "does not qualify"}   [{len(_common)} paired rows]')
    print('\nIf several qualify: the most parsimonious wins (fewest factors, then fewest HSGP timescales).')

<br>

## 4 &emsp; Holdout Protocol (sealed)

`2024/25` is scored **once**, for the committed spec only, after the collaborator review of the
verdict above: set `COMMITTED = devVersion = '<winner>'`, `RUN_HOLDOUT = True`, `SMOKE = False`,
and run top to bottom — the USER INTERACTION cell re-points the window (train ≤ 2023/24 →
score 2024/25) and the Export cell writes the `__EW_HOLDOUT` ledger. **Report the number as-is;
re-running the holdout for a second spec is a protocol violation** — the number it produced
would not be a holdout number. When 2025/26 data lands, it becomes the pristine second holdout.

<br>

## 5 &emsp; Export — Production Model

The SFMMO pattern (`SFMMOwm__final_EW.ipynb`, "Export — Production Model"): the *same*
notebook that selected the spec also fits it on the full window and writes the bundle the
serving script loads. Set `RUN_PRODUCTION = True` + `devVersion = COMMITTED` in USER
INTERACTION, run the Grand Loop (it fits and stops), then run this cell.

The bundle keys match **exactly** what `006_040__Predictions_ScoringProb` reads
(`idata`, `factor_standardize`, `data`, `indices`, `model`, `params_HSGP`), plus the ELO
state C_ELO needs at serving time.

In [ ]:
# ================================== Export -- Production Model ================================== #
#
# Writes 10_data/01_Models/SFM_II_Final{devVersion}_scaleCS__{train_end}.pkl
# Consumed by 006_040__Predictions_ScoringProb (set SFM_model__NAME to the file stem).
#
# Contract (verified against 006_040 lines 222-485):
#   idata | factor_standardize | indices['players']['unique'] | model | params_HSGP
# NOTE: the OG bundle also pickles 'data' = the whole training frame. Deliberately DROPPED
# here (the SFMMOwm bundle carries index maps, not data): every consumer re-reads
# data_byPlayer__SFM_II.csv anyway, so the copy was dead weight AND a second source of truth
# that can drift from the CSV. A fingerprint travels instead, checked at serving time.
# Plus, NEW for C_ELO and required by any serving path:
#   elo  -> final per-league ratings + the conventions needed to roll them forward,
#           and the per-match table for audit. 006_040 must gain an ELO step before it
#           can serve C_ELO (see FINDINGS_2026_REVISION.md §5 production port).

assert RUN_PRODUCTION, 'set RUN_PRODUCTION = True (and devVersion = COMMITTED) first'
assert not SMOKE, 'SMOKE run -- a 200/200-draw bundle must never reach production. Set SMOKE = False.'
assert devVersion == COMMITTED == 'C_ELO', 'production serves the COMMITTED spec only'
assert 'prod' in dir(), 'run the Grand Loop first -- it fits the production model'

import pickle
import cloudpickle

# --- the harness ALWAYS standardizes cross-sectionally (season x gameday, factor-wise);
# --- unlike the dev notebook there is no do__scaleCS switch to consult.
scale__type = '_scaleCS'
_stem = f'SFM_II_Final{devVersion}{scale__type}__{PRODUCTION_TRAIN_END[2:4]}{PRODUCTION_TRAIN_END[-2:]}'
pickle_filepath = f'{directory}/10_data/01_Models/{_stem}.pkl'

dict_to_save = {
    # --- the 006_040 contract -------------------------------------------------------
    'model': prod['model'],
    'idata': prod['idata'],
    'factor_standardize': factors_CS,           # --- CS-standardized factor list (incl. elo_diff)
    'params_HSGP': {'lengthscales': {'short': prod['ls_short'], 'medium': prod['ls_medium'],
                                     'long': prod['ls_long']},
                    'basis_vectors': {'within': prod['m_within'], 'long': prod['m_long']},
                    'scaling_factors': {'within': prod['c_within'], 'long': prod['c_long']},
                    'devVersion': devVersion},
    'indices': {'seasons':  {'unique': prod['unique_seasons'],  'idx': []},
                'players':  {'unique': prod['players_ordered'], 'idx': prod['player_idx']},
                'gamedays': {'unique': prod['unique_gamedays'], 'idx': prod['gameday_idx']}},
    # --- 'data' intentionally NOT pickled (see the note above); fingerprint instead:
    'data_contract': {'source': '10_data/106_Website/data_byPlayer__SFM_II.csv',
                      'train_end': PRODUCTION_TRAIN_END,
                      'n_rows': int(len(prod['train_df'])),
                      'seasons': sorted(prod['train_df'].season.unique().tolist()),
                      'goal_cats_share': prod['train_df']['goals_cats']
                                         .value_counts(normalize=True).sort_index().round(6).to_dict()},
    # --- everything the serving side needs to rebuild the spec faithfully ------------
    'devVersion': devVersion,
    'cfg': cfg,
    'factors_team': factors_team,
    'factors_player': factors_player,
    'scaling': {'grouping': ['season', 'gameday'], 'factor_wise': True},
    'elo': {'ratings_final': elo_state,          # --- {league: {team: rating}} after the last match
            'K': 20, 'home_adv': 50,
            'season_start': {'returning': (0.75, 1500), 'promoted': 1300},
            'match_table': elo_match_table},
    'provenance': {'train_end': PRODUCTION_TRAIN_END, 'n_rows': int(len(prod['train_df'])),
                   'n_players': int(len(prod['players_ordered'])),
                   'seed': seed, 'n_chains': N_chains, 'target_accept': TARGET_ACCEPT,
                   'noncenter_zsn': NONCENTER_ZSN,
                   'selection': 'closed 2026-08-13; holdout 2024/25 burned 2026-08-14 (+3.56% skill)'},
}

with open(pickle_filepath, 'wb') as f:
    cloudpickle.dump(dict_to_save, f)


# ---------------------------- LIGHT bundle (pure-NumPy serving) ----------------------------
# A plain-pickle artifact loadable WITHOUT pymc -- the SFMMOwm philosophy taken to its end.
# VALIDATED 2026-08-14 against a real graph reference (sfmII venv, pymc 5.26.1):
#   same players: per-draw probs match to 4.4e-16 | new players: within MC tolerance.
# f_within / f_long live on FIXED grids and are exported FROM THE GRAPH here -- serving
# never re-derives GP math. Golden rows travel inside: serving recomputes them and asserts
# equality before predicting anything.

def _softplus_np(x):
    return np.logaddexp(0.0, x)

def _ordered_probs_np(eta_s, cut_s):
    cdf = 1.0 / (1.0 + np.exp(-(cut_s - eta_s[..., None])))
    return np.concatenate([cdf[..., :1], np.diff(cdf, axis=-1), 1.0 - cdf[..., -1:]], axis=-1)

def _flat(da):
    return np.ascontiguousarray(da.stack(s=('chain', 'draw')).transpose('s', ...).to_numpy())

with prod['model']:
    _fpp = pm.sample_posterior_predictive(prod['idata'], var_names=['f_within', 'f_long'],
                                          predictions=True, random_seed=seed)

_post = prod['idata'].posterior
_draws = {v: _flat(_post[v]) for v in
          ['baseline', 'player_effect_diversity', 'player_effect_raw', 'beta__team',
           'delta_mean', 'delta_sigma', 'delta_player']}
_fw = _flat(_fpp['predictions']['f_within'])
_fl = _flat(_fpp['predictions']['f_long'])
_Np = len(prod['players_ordered'])

# --- golden rows: 64 training rows, probs computed by the SAME algebra serving will run
_grows = np.sort(np.random.default_rng(seed).choice(len(prod['train_df']), 64, replace=False))
_g_pl = prod['player_idx'][_grows]
_g_gd = prod['gameday_idx'][_grows]
_g_ss = prod['train_df']['season_nbr'].to_numpy()[_grows]
_g_X  = prod['train_df'][factors_team].to_numpy()[_grows].astype(np.float64)

_bsig = np.sqrt(25 + _draws['player_effect_diversity']**2 / _Np)
_pe   = _bsig[:, None] * _draws['baseline'][:, None] + _draws['player_effect_raw']
_dl   = _draws['delta_mean'][:, None, :] + _draws['delta_sigma'][:, None, :] * _draws['delta_player']
_cut  = np.concatenate([np.full((_pe.shape[0], _Np, 1), 4.0),
                        4.0 + np.cumsum(_softplus_np(_dl), axis=-1)], axis=-1)
_g_eta = (_pe[:, _g_pl] + _fw[:, _g_gd] + _fl[:, _g_ss]
          + np.einsum('nf,sf->sn', _g_X, _draws['beta__team']))
_g_probs = _ordered_probs_np(_g_eta, _cut[:, _g_pl])

light_filepath = pickle_filepath.replace('.pkl', '__LIGHT.pkl')
with open(light_filepath, 'wb') as f:
    pickle.dump({
        'draws': _draws,
        'f_within': _fw, 'f_long': _fl,
        'n_players_train': _Np, 'intercept_sigma': 5, 'cutpoint_offset': 4.0,
        'players_ordered': prod['players_ordered'],
        'unique_gamedays': prod['unique_gamedays'], 'unique_seasons': prod['unique_seasons'],
        'factors_team': factors_team, 'factor_standardize': factors_CS,
        'scaling': dict_to_save['scaling'], 'elo': dict_to_save['elo'],
        'data_contract': dict_to_save['data_contract'], 'provenance': dict_to_save['provenance'],
        'golden': {'player_codes': _g_pl, 'gd_idx': _g_gd, 'season_idx': _g_ss,
                   'X': _g_X, 'probs': _g_probs},
    }, f, protocol=4)

_mb = sum(a.nbytes for a in _draws.values()) + _fw.nbytes + _fl.nbytes
print(f'LIGHT bundle written: {light_filepath}  (~{_mb/1e6:.0f} MB of draws; plain pickle, no pymc needed)')
del _fpp, _pe, _cut, _g_probs

print(f'Production bundle written: {pickle_filepath}')
print(f'  spec {devVersion} | trained through {PRODUCTION_TRAIN_END} | '
      f'{len(prod["train_df"]):,} rows | {len(prod["players_ordered"]):,} players')
print(f'  factor_standardize: {factors_CS}')
print(f'  factors_team: {factors_team} | factors_player: {factors_player}')
print(f'  elo state: {len(elo_state)} leagues, '
      f'{sum(len(v) for v in elo_state.values())} teams carried forward')
print(f'\n  -> in 006_040__Predictions_ScoringProb set  SFM_model__NAME = "{_stem}"')
print('  -> 006_040 still needs the ELO step ported before it can serve C_ELO.')
